# Блок 2. Практика: Vector

В этом notebook мы:
1. Запустим Vector в Docker
2. Научимся читать и писать конфигурацию
3. Потренируемся с VRL
4. Подготовимся к заданию

## Часть 1: Запуск Vector

Vector распространяется как Docker-образ. Запустим его и проверим, что всё работает:

In [ ]:
# Проверим версию Vector
!docker run --rm timberio/vector:latest-alpine vector --version

## Часть 2: Минимальная конфигурация

Создадим простейший конфиг: генерируем тестовые события и выводим в консоль.

In [ ]:
%%writefile data/demo_config.toml
# Источник: генератор тестовых событий
[sources.demo]
type = "demo_logs"
format = "json"
interval = 1.0  # одно событие в секунду

# Приёмник: вывод в консоль
[sinks.console]
type = "console"
inputs = ["demo"]
encoding.codec = "json"

In [ ]:
# Запустим Vector с этим конфигом на 5 секунд
# --rm удалит контейнер после остановки
# -v монтируем директорию с конфигом
!timeout 5 docker run --rm \
    -v $(pwd)/data:/data \
    timberio/vector:latest-alpine \
    vector --config /data/demo_config.toml 2>/dev/null || true

Видим JSON-события, которые Vector генерирует и выводит. Каждое событие содержит:
- `host` — откуда пришло событие
- `message` — текст сообщения  
- `timestamp` — время

Это базовая структура, которую Vector использует для всех событий.

## Часть 3: Transforms — обработка данных

Главная сила Vector — трансформации. Добавим обработку с VRL:

In [ ]:
%%writefile data/transform_config.toml
[sources.demo]
type = "demo_logs"
format = "json"
interval = 1.0

# Трансформация: добавляем новые поля
[transforms.enrich]
type = "remap"
inputs = ["demo"]
source = '''
# Добавляем метку окружения
.environment = "development"

# Добавляем время обработки
.processed_at = now()

# Извлекаем длину сообщения
.message_length = length(.message)
'''

[sinks.console]
type = "console"
inputs = ["enrich"]  # Берём данные из трансформа, не из источника!
encoding.codec = "json"

In [ ]:
!timeout 5 docker run --rm \
    -v $(pwd)/data:/data \
    timberio/vector:latest-alpine \
    vector --config /data/transform_config.toml 2>/dev/null || true

Теперь в каждом событии есть новые поля: `environment`, `processed_at`, `message_length`.

## Часть 4: Чтение из файла

В реальности мы читаем данные из файлов. Попробуем прочитать наш тестовый nginx-лог:

In [ ]:
# Посмотрим на формат логов
!head -3 data/nginx_access.log

Это стандартный combined-формат nginx:
```
IP - - [timestamp] "method path protocol" status size "referer" "user_agent"
```

Создадим конфиг для чтения файла:

In [ ]:
%%writefile data/file_config.toml
[sources.nginx]
type = "file"
include = ["/data/nginx_access.log"]
read_from = "beginning"

[sinks.console]
type = "console"
inputs = ["nginx"]
encoding.codec = "json"

In [ ]:
!timeout 3 docker run --rm \
    -v $(pwd)/data:/data \
    timberio/vector:latest-alpine \
    vector --config /data/file_config.toml 2>/dev/null | head -5

Vector прочитал файл. Обратите внимание:
- Весь лог попал в поле `message`
- Добавлено поле `file` с путём к файлу
- Данные не распарсены — это просто строки

Чтобы извлечь полезную информацию, нужен парсинг.

## Часть 5: VRL Playground

Прежде чем писать конфиги, удобно тестировать VRL-выражения. Vector имеет встроенную команду `vrl`:

In [ ]:
# Тестируем VRL-выражения
# Команда vrl принимает программу и входные данные

!echo '{"status": "403", "path": "/admin"}' | docker run --rm -i \
    timberio/vector:latest-alpine \
    vrl '.severity = if to_int!(.status) >= 400 { "error" } else { "info" }; .'

In [ ]:
# Тестируем парсинг regex — простой пример: извлекаем IP и статус
import subprocess
log_line = '192.168.1.100 - - [15/Jan/2024:10:15:32 +0000] "GET /api/users HTTP/1.1" 200 1234'

result = subprocess.run(
    ['docker', 'run', '--rm', '-i', 'timberio/vector:latest-alpine', 'vrl',
     r'. |= parse_regex!(.message, r"^(?P<ip>\S+) .* (?P<status>\d+) \d+$"); .'],
    input=f'{{"message": "{log_line}"}}',
    capture_output=True, text=True
)
print(result.stdout)

## Часть 6: Шпаргалка по VRL

**Парсинг:**
- `parse_json!(.field)` — парсинг JSON
- `parse_regex!(.field, r"pattern")` — regex с именованными группами
- `parse_syslog!(.field)` — syslog-формат
- `parse_key_value!(.field)` — формат key=value

**Строки:**
- `contains(.field, "substring")` — проверка подстроки
- `starts_with(.field, "prefix")` — проверка начала
- `downcase(.field)` / `upcase(.field)` — регистр
- `replace(.field, "old", "new")` — замена

**Преобразование типов:**
- `to_int!(.field)` — в число
- `to_string(.field)` — в строку

**Время:**
- `now()` — текущее время
- `parse_timestamp!(.field, "%Y-%m-%d")` — парсинг даты

**Условия:**
- `exists(.field)` — поле существует
- `if condition { ... } else { ... }` — условие

## Часть 7: Проверка конфигурации

Vector умеет проверять конфиг без запуска:

In [ ]:
# Проверяем конфиг на ошибки
!docker run --rm \
    -v $(pwd)/data:/data \
    timberio/vector:latest-alpine \
    vector validate /data/transform_config.toml

In [ ]:
%%writefile data/broken_config.toml
[sources.demo]
type = "demo_logs"

[sinks.console]
type = "console"
inputs = ["wrong_name"]
encoding.codec = "json"

In [ ]:
# Проверяем конфиг с ошибкой
!docker run --rm \
    -v $(pwd)/data:/data \
    timberio/vector:latest-alpine \
    vector validate /data/broken_config.toml

Vector показывает ошибку: input `wrong_name` не найден. Всегда проверяйте конфиг перед запуском!

## Часть 8: Подготовка к заданию

Для задания вам нужно:

1. **Распарсить nginx-логи** с помощью regex. Формат combined:
   ```
   IP - - [timestamp] "method path protocol" status size "referer" "user_agent"
   ```
   
   Подсказка: regex должен извлекать именованные группы `(?P<name>pattern)`

2. **Добавить поле severity** на основе статуса:
   - status >= 400 → `error`
   - иначе → `info`

3. **Для второго задания** — отправить в PostgreSQL. Вам понадобится:
   - Запущенная база из Блока 1
   - Sink типа `postgresql` (смотрите документацию Vector)

In [ ]:
# Ваш эксперимент с VRL
# Попробуйте написать regex для парсинга nginx-логов

# Пример запуска:
# !echo '{"message": "..."}' | docker run --rm -i timberio/vector:latest-alpine vrl 'ваш_код'

## Полезные команды

```bash
# Проверить конфиг
docker run --rm -v $(pwd):/data timberio/vector:latest-alpine vector validate /data/config.toml

# Запустить Vector
docker run --rm -v $(pwd):/data timberio/vector:latest-alpine vector --config /data/config.toml

# Интерактивный VRL
docker run --rm -it timberio/vector:latest-alpine vrl

# Тестировать VRL с данными
echo '{"field": "value"}' | docker run --rm -i timberio/vector:latest-alpine vrl '.new_field = "added"; .'
```

---

Теперь вы готовы к выполнению заданий! Создайте файл `vector.toml` в директории `lesson02/` и настройте pipeline для обработки nginx-логов.

In [ ]:
# Тестируем парсинг regex
# Простой пример: извлекаем IP и статус

import subprocess
log_line = '192.168.1.100 - - [15/Jan/2024:10:15:32 +0000] "GET /api/users HTTP/1.1" 200 1234 "-" "Mozilla/5.0"'

result = subprocess.run(
    ['docker', 'run', '--rm', '-i', 'timberio/vector:latest-alpine', 'vrl',
     r'. |= parse_regex!(.message, r"^(?P<ip>\S+) .* (?P<status>\d+) \d+"); .'],
    input=f'{{"message": "{log_line}"}}',
    capture_output=True, text=True
)
print(result.stdout)